# Experiment 03 — Feature Ablation Study

**Goal:** Determine the incremental predictive value of each feature group for STLF at H=24.

| Experiment | Features | Lookback |
|------------|----------|----------|
| E0 | Load only (baseline) | 24 h |
| E1 | Load + Weather | 24 h |
| E2 | Load + Temporal (cyclical hour/dow/month) | 24 h |
| E3 | Load + Lag-24 + Lag-168 | 24 h |
| E4 | Load + Weather + Temporal + Lags | 24 h |
| E5 | Load + Weather + Temporal + Lags | 168 h |

**Models:** Persistence / Daily Naive / Weekly Naive / Linear Regression / LSTM / BiLSTM  
**Model selection:** Validation MAPE only — test is evaluated ONCE for the best configuration.

> **E6 skipped** — dataset contains only observed weather; no legitimate future forecasts exist.

In [ ]:
# ── Cell 1: Environment Setup ────────────────────────────────────
from pathlib import Path
import subprocess, sys

PROJECT_ROOT = Path("/kaggle/working/stlf-entso-2026")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/AlvinHarist/stlf-entso-2026.git",
         str(PROJECT_ROOT)],
        check=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

In [ ]:
# ── Cell 2: Imports & Configuration ──────────────────────────────
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression

from src.utils.seed import set_seed
from src.data.load_data import load_dataset
from src.data.preprocessing import (
    chronological_split,
    fit_preprocessor,
    transform_data,
    inverse_y,
    create_temporal_features,
    create_lag_features,
    TEMPORAL_FEATURE_COLS,
    LAG_FEATURE_COLS_DEFAULT,
)
from src.data.windowing import create_train_windows, create_evaluation_windows
from src.models.lstm import build_lstm
from src.models.bilstm import build_bilstm
from src.training.trainer import train_model
from src.evaluation.point_metrics import (
    compute_all_metrics,
    compute_naive_baselines,
    mae, rmse, mape, smape,
)

CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

SEED       = config["seed"]
TARGET_COL = config["data"]["target_col"]
WEATHER    = config["data"]["weather_features"]
UNITS      = config["model"]["units"]
DROPOUT    = config["model"]["dropout"]
LR         = config["model"]["learning_rate"]
EPOCHS     = config["training"]["epochs"]
BATCH_SIZE = config["training"]["batch_size"]
PATIENCE   = config["training"]["patience"]
HORIZON    = 24   # Fixed for all feature-ablation experiments

# Kaggle data path
KAGGLE_DATA_DIR = Path("/kaggle/input/stlf-entso-2026")
DATA_PATH = None
if KAGGLE_DATA_DIR.exists():
    for p in KAGGLE_DATA_DIR.rglob("*.csv"):
        if "combined_AT" in p.name:
            DATA_PATH = p
            break
if DATA_PATH is None:
    fallback = PROJECT_ROOT / config["data"]["path"]
    if fallback.exists():
        DATA_PATH = fallback
if DATA_PATH is None:
    fallback = PROJECT_ROOT / "df_combined_AT.csv"
    if fallback.exists():
        DATA_PATH = fallback
if DATA_PATH is None:
    raise FileNotFoundError("Cannot locate df_combined_AT.csv")

RESULTS_DIR = PROJECT_ROOT / "results" / "feature_ablation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset   : {DATA_PATH}")
print(f"Results   : {RESULTS_DIR}")
print(f"Horizon   : {HORIZON} h")
print(f"Seed      : {SEED}")

In [ ]:
# ── Cell 3: Load Data & Feature Engineering ───────────────────────
# Load raw dataset
df_raw = load_dataset(DATA_PATH, timestamp_col=config["data"]["timestamp_col"])
print(f"Raw dataset: {df_raw.shape}  {df_raw.index.min()} --> {df_raw.index.max()}")

# Step 1: Add temporal cyclical features (uses only row's own timestamp)
df_feat = create_temporal_features(df_raw)

# Step 2: Add lag features on the FULL DataFrame before splitting
# lag_k[t] = target[t-k] — strictly backwards, no leakage
df_feat = create_lag_features(df_feat, TARGET_COL, lags=[24, 168])

print(f"Feature-engineered dataset: {df_feat.shape}")
print(f"Columns: {list(df_feat.columns)}")

# Sanity check: lag_168[t] must equal target[t-168]
sample_t = 200
assert abs(df_feat["lag_168"].iloc[sample_t] - df_feat[TARGET_COL].iloc[sample_t - 168]) < 1e-6, \
    "Lag feature sanity check failed!"
print("Lag sanity check: PASSED")

In [ ]:
# ── Cell 4: Chronological Split ──────────────────────────────────
train_df, val_df, test_df = chronological_split(
    df_feat,
    train_ratio=config["split"]["train_ratio"],
    val_ratio=config["split"]["val_ratio"],
)

# Store split boundaries for naive baselines
full_series_mw = df_feat[TARGET_COL].values
val_start_idx  = len(train_df)
test_start_idx = len(train_df) + len(val_df)

# Leakage assertion
assert train_df.index.max() < val_df.index.min(), "Train/val boundary violation!"
assert val_df.index.max() < test_df.index.min(), "Val/test boundary violation!"
print("Split boundary assertions: PASSED")

In [ ]:
# ── Cell 5: Helper Function ───────────────────────────────────────
def run_experiment(
    exp_name,
    feature_cols,
    lookback,
    model_name,
    evaluate_on_test=False,
):
    """Run one (experiment, model) combination.
    
    Returns a dict with val and (optionally) test metrics, plus per-window
    DataFrames.
    
    model_name: 'LinearRegression' | 'LSTM' | 'BiLSTM'
    evaluate_on_test: set True only for the final selected config.
    """
    set_seed(SEED)

    # ---- Preprocessing (train-only fit) --------------------------------
    use_yj = any(c in feature_cols for c in config["preprocessing"]["skewed_cols"])
    skewed = config["preprocessing"]["skewed_cols"] if use_yj else None

    preprocessor = fit_preprocessor(
        train_df, TARGET_COL, feature_cols,
        skewed_cols=skewed, use_yeojohnson=use_yj,
    )

    X_tr, y_tr = transform_data(train_df, preprocessor)
    X_va, y_va = transform_data(val_df,   preprocessor)
    X_te, y_te = transform_data(test_df,  preprocessor)

    # ---- Windowing ------------------------------------------------------
    Xw_tr, yw_tr = create_train_windows(X_tr, y_tr, lookback, HORIZON)
    Xw_va, yw_va = create_evaluation_windows(X_va, y_va, X_tr, y_tr, lookback, HORIZON)
    Xw_te, yw_te = create_evaluation_windows(X_te, y_te, X_va, y_va, lookback, HORIZON)

    print(f"  [{exp_name}/{model_name}] Windows  tr={Xw_tr.shape}  va={Xw_va.shape}  te={Xw_te.shape}")

    # ---- Build & Train --------------------------------------------------
    best_epoch = "N/A"
    if model_name == "LinearRegression":
        clf = LinearRegression()
        clf.fit(Xw_tr.reshape(Xw_tr.shape[0], -1), yw_tr)
        pred_va_scaled = clf.predict(Xw_va.reshape(Xw_va.shape[0], -1))
        pred_te_scaled = clf.predict(Xw_te.reshape(Xw_te.shape[0], -1))
    else:
        n_feat = Xw_tr.shape[2]
        if model_name == "LSTM":
            model = build_lstm(lookback, n_feat, HORIZON, UNITS, DROPOUT, LR)
        else:
            model = build_bilstm(lookback, n_feat, HORIZON, UNITS, DROPOUT, LR)
        tr_res = train_model(
            model, Xw_tr, yw_tr, Xw_va, yw_va,
            EPOCHS, BATCH_SIZE, PATIENCE, verbose=1,
        )
        best_epoch = tr_res["best_epoch"]
        pred_va_scaled = model.predict(Xw_va)
        pred_te_scaled = model.predict(Xw_te)

    # ---- Inverse Transform & Metrics ------------------------------------
    def _get_per_window_df(pred_mw, actual_mw):
        rows = []
        for i in range(len(pred_mw)):
            yt, yp = actual_mw[i], pred_mw[i]
            row = {
                "window_index": i,
                "mae":   mae(yt, yp),
                "rmse":  rmse(yt, yp),
                "mape":  mape(yt, yp),
                "smape": smape(yt, yp),
            }
            for h in range(HORIZON):
                row[f"actual_h{h+1}"] = yt[h]
                row[f"pred_h{h+1}"]   = yp[h]
            rows.append(row)
        return pd.DataFrame(rows)

    pred_va_mw  = inverse_y(pred_va_scaled, preprocessor)
    actual_va_mw = inverse_y(yw_va,          preprocessor)
    val_mets  = compute_all_metrics(actual_va_mw, pred_va_mw)
    df_val_pw = _get_per_window_df(pred_va_mw, actual_va_mw)

    result = {
        "exp": exp_name,
        "model": model_name,
        "feature_cols": feature_cols,
        "lookback": lookback,
        "best_epoch": best_epoch,
        "val_metrics": val_mets,
        "df_val_pw": df_val_pw,
        "actual_va_mw": actual_va_mw,
        "pred_va_mw": pred_va_mw,
    }

    # Save val CSV
    agg_rec = {"experiment": exp_name, "model": model_name,
               "lookback": lookback, "best_epoch": best_epoch}
    agg_rec.update(val_mets)
    pd.DataFrame([agg_rec]).to_csv(
        RESULTS_DIR / f"{exp_name}_{model_name}_val_metrics.csv", index=False)
    df_val_pw.to_csv(
        RESULTS_DIR / f"{exp_name}_{model_name}_per_window_val.csv", index=False)

    if evaluate_on_test:
        pred_te_mw   = inverse_y(pred_te_scaled, preprocessor)
        actual_te_mw = inverse_y(yw_te,           preprocessor)
        test_mets  = compute_all_metrics(actual_te_mw, pred_te_mw)
        df_test_pw = _get_per_window_df(pred_te_mw, actual_te_mw)

        te_rec = {"experiment": exp_name, "model": model_name,
                  "lookback": lookback, "best_epoch": best_epoch}
        te_rec.update(test_mets)
        pd.DataFrame([te_rec]).to_csv(
            RESULTS_DIR / f"{exp_name}_{model_name}_test_metrics.csv", index=False)
        df_test_pw.to_csv(
            RESULTS_DIR / f"{exp_name}_{model_name}_per_window_test.csv", index=False)

        result["test_metrics"] = test_mets
        result["df_test_pw"]   = df_test_pw
        result["actual_te_mw"] = actual_te_mw
        result["pred_te_mw"]   = pred_te_mw
        print(f"  [{exp_name}/{model_name}] TEST  MAE={test_mets['MAE']:.2f}  MAPE={test_mets['MAPE']:.2f}%")

    print(f"  [{exp_name}/{model_name}] VAL   MAE={val_mets['MAE']:.2f}  MAPE={val_mets['MAPE']:.2f}%")
    return result


def run_naive_baselines(split="val"):
    """Compute naive baselines on the same forecast windows."""
    if split == "val":
        start_idx = val_start_idx
        n_windows = len(val_df) - HORIZON + 1
    else:
        start_idx = test_start_idx
        n_windows = len(test_df) - HORIZON + 1

    # Build actual targets for the chosen split (load-only, no scaling needed
    # since naive baselines operate on original scale)
    results = {}
    y_actual = []
    for w in range(n_windows):
        origin = start_idx + w
        y_actual.append(full_series_mw[origin:origin + HORIZON])
    y_actual = np.array(y_actual)

    baselines = compute_naive_baselines(y_actual, full_series_mw, start_idx, HORIZON)

    for name, mets in baselines.items():
        rec = {"experiment": name, "model": "naive",
               "lookback": "N/A", "best_epoch": "N/A"}
        rec.update(mets)
        pd.DataFrame([rec]).to_csv(
            RESULTS_DIR / f"{name}_{split}_metrics.csv", index=False)

    return baselines

print("Helper functions defined.")

In [ ]:
# ── Cell 6: Experiment Configurations ────────────────────────────
EXPERIMENTS = [
    {
        "name": "E0_load_only",
        "feature_cols": [TARGET_COL],
        "lookback": 24,
        "description": "Load only (univariate baseline)",
    },
    {
        "name": "E1_weather",
        "feature_cols": [TARGET_COL] + WEATHER,
        "lookback": 24,
        "description": "Load + weather variables",
    },
    {
        "name": "E2_temporal",
        "feature_cols": [TARGET_COL] + TEMPORAL_FEATURE_COLS,
        "lookback": 24,
        "description": "Load + cyclical temporal features (hour/dow/month)",
    },
    {
        "name": "E3_lags",
        "feature_cols": [TARGET_COL] + LAG_FEATURE_COLS_DEFAULT,
        "lookback": 24,
        "description": "Load + lag_24 + lag_168",
    },
    {
        "name": "E4_full_24",
        "feature_cols": [TARGET_COL] + WEATHER + TEMPORAL_FEATURE_COLS + LAG_FEATURE_COLS_DEFAULT,
        "lookback": 24,
        "description": "Load + weather + temporal + lags (full, LB=24)",
    },
    {
        "name": "E5_full_168",
        "feature_cols": [TARGET_COL] + WEATHER + TEMPORAL_FEATURE_COLS + LAG_FEATURE_COLS_DEFAULT,
        "lookback": 168,
        "description": "Load + weather + temporal + lags (full, LB=168)",
    },
]

NEURAL_MODELS = ["LSTM", "BiLSTM"]

print("Experiment configurations:")
for e in EXPERIMENTS:
    print(f"  {e['name']:15s} | LB={e['lookback']:3d} | {len(e['feature_cols'])} features | {e['description']}")

In [ ]:
# ── Cell 7: Run All Experiments on VALIDATION ─────────────────────
# Model selection is done on VALIDATION — test set is NOT touched here.

# 7a. Naive baselines (no training, run once)
print("=" * 60)
print("Running naive baselines on VALIDATION...")
print("=" * 60)
val_baselines = run_naive_baselines(split="val")
for name, m in val_baselines.items():
    print(f"  {name:<20s}  MAE={m['MAE']:.2f}  MAPE={m['MAPE']:.2f}%")

# 7b. LR + Neural models across all experiments
all_val_results = {}   # key: (exp_name, model_name) -> result dict

for exp in EXPERIMENTS:
    # Linear Regression
    print(f"\n{'='*60}")
    print(f" {exp['name']} | LinearRegression")
    print(f"{'='*60}")
    res = run_experiment(exp["name"], exp["feature_cols"], exp["lookback"], "LinearRegression")
    all_val_results[(exp["name"], "LinearRegression")] = res

    for mname in NEURAL_MODELS:
        print(f"\n{'='*60}")
        print(f" {exp['name']} | {mname}")
        print(f"{'='*60}")
        res = run_experiment(exp["name"], exp["feature_cols"], exp["lookback"], mname)
        all_val_results[(exp["name"], mname)] = res

print("\nValidation experiments complete.")

In [ ]:
# ── Cell 8: Model Selection & Final Test Evaluation ───────────────
# Select best config by VALIDATION MAPE only.
# Then evaluate on TEST exactly once.

best_key   = min(all_val_results, key=lambda k: all_val_results[k]["val_metrics"]["MAPE"])
best_result = all_val_results[best_key]
best_exp_name = best_key[0]
best_model    = best_key[1]

print("=" * 60)
print("MODEL SELECTION (Validation MAPE)")
print("=" * 60)
print(f"  Best config : {best_exp_name} / {best_model}")
print(f"  Val MAPE    : {best_result['val_metrics']['MAPE']:.4f}%")
print()

# Retrieve the matching experiment config
best_exp_cfg = next(e for e in EXPERIMENTS if e["name"] == best_exp_name)

# Final evaluation on TEST — run once, clearly labeled
print("Running FINAL TEST evaluation for best config...")
final_result = run_experiment(
    best_exp_cfg["name"],
    best_exp_cfg["feature_cols"],
    best_exp_cfg["lookback"],
    best_model,
    evaluate_on_test=True,
)

# Also run naive baselines on TEST for comparison
print("\nNaive baselines on TEST...")
test_baselines = run_naive_baselines(split="test")

print("\n" + "=" * 60)
print("FINAL TEST RESULTS (best config)")
print("=" * 60)
tm = final_result["test_metrics"]
print(f"  {best_exp_name} / {best_model}")
print(f"  MAE={tm['MAE']:.2f}  RMSE={tm['RMSE']:.2f}  MAPE={tm['MAPE']:.2f}%  sMAPE={tm['sMAPE']:.2f}%")
for name, m in test_baselines.items():
    print(f"  {name:<20s}  MAE={m['MAE']:.2f}  MAPE={m['MAPE']:.2f}%")

In [ ]:
# ── Cell 9: Summary Tables ────────────────────────────────────────

# Table 1: Full validation MAPE matrix (Experiment x Model)
exp_names   = [e["name"] for e in EXPERIMENTS]
model_names = ["LinearRegression"] + NEURAL_MODELS

val_mape_matrix = pd.DataFrame(index=exp_names, columns=model_names, dtype=float)
for (en, mn), res in all_val_results.items():
    val_mape_matrix.loc[en, mn] = res["val_metrics"]["MAPE"]

# Add naive baselines as extra rows
for name, m in val_baselines.items():
    val_mape_matrix.loc[name, "LinearRegression"] = m["MAPE"]

val_mape_matrix.to_csv(RESULTS_DIR / "feature_ablation_val_mape_matrix.csv")

print("\nValidation MAPE Matrix (%)")
print("=" * 70)
print(val_mape_matrix.round(4).to_string())

# Table 2: Full validation metrics per experiment/model
val_summary_rows = []
for (en, mn), res in all_val_results.items():
    row = {"experiment": en, "model": mn, "lookback": res["lookback"]}
    row.update(res["val_metrics"])
    val_summary_rows.append(row)
df_val_summary = pd.DataFrame(val_summary_rows)
df_val_summary.to_csv(RESULTS_DIR / "feature_ablation_val_summary.csv", index=False)

print("\n\nFull Validation Metrics Table")
print("=" * 70)
print(df_val_summary.sort_values("MAPE").to_string(index=False))

# Table 3: Final test results
test_summary_rows = []
best_row = {"experiment": best_exp_name, "model": best_model,
            "lookback": best_result["lookback"], "split": "test (final)"}
best_row.update(final_result["test_metrics"])
test_summary_rows.append(best_row)
for name, m in test_baselines.items():
    row = {"experiment": name, "model": "naive", "lookback": "N/A", "split": "test (final)"}
    row.update(m)
    test_summary_rows.append(row)
df_test_summary = pd.DataFrame(test_summary_rows)
df_test_summary.to_csv(RESULTS_DIR / "feature_ablation_test_summary.csv", index=False)

print("\n\nFinal Test Results (best config only)")
print("=" * 70)
print(df_test_summary.to_string(index=False))

In [ ]:
# ── Cell 10: Visualizations ──────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = sns.color_palette("Set2", 6)

# ── 10A: Validation MAPE bar chart by experiment and model ───────────────
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(EXPERIMENTS))
bar_w = 0.22
for j, mname in enumerate(["LinearRegression", "LSTM", "BiLSTM"]):
    vals = [all_val_results[(e["name"], mname)]["val_metrics"]["MAPE"]
            for e in EXPERIMENTS]
    ax.bar(x + j * bar_w, vals, width=bar_w, label=mname, color=COLORS[j])

# Add naive weekly baseline
ax.axhline(val_baselines["weekly_naive"]["MAPE"], color="red", linestyle="--",
           linewidth=1.5, label="Weekly Naive")
ax.set_xticks(x + bar_w)
ax.set_xticklabels([e["name"] for e in EXPERIMENTS], rotation=15, ha="right")
ax.set_ylabel("MAPE (%)")
ax.set_title("Feature Ablation — Validation MAPE by Experiment & Model")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "val_mape_bar.png", dpi=150)
plt.show()

# ── 10B: Time-series Best vs Worst vs True for h=1 on validation ─────────
worst_key  = max(all_val_results, key=lambda k: all_val_results[k]["val_metrics"]["MAPE"])
best_key_v = min(all_val_results, key=lambda k: all_val_results[k]["val_metrics"]["MAPE"])

best_pred_v  = all_val_results[best_key_v]["pred_va_mw"]
worst_pred_v = all_val_results[worst_key]["pred_va_mw"]
actual_v     = all_val_results[best_key_v]["actual_va_mw"]

n_plot = min(168, len(actual_v))
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(actual_v[:n_plot, 0], label="True Load", color="black", linewidth=2)
ax.plot(best_pred_v[:n_plot, 0], label=f"Best ({best_key_v[0]}/{best_key_v[1]})",
        color="#2ca02c", linestyle="--", linewidth=1.5)
ax.plot(worst_pred_v[:n_plot, 0], label=f"Worst ({worst_key[0]}/{worst_key[1]})",
        color="#d62728", linestyle=":", alpha=0.8, linewidth=1.5)
ax.set_title("Validation — True Load vs Best & Worst Config (h=1, First 7 Days)", fontsize=14)
ax.set_xlabel("Window Index")
ax.set_ylabel("Load (MW)")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "best_vs_worst_timeseries.png", dpi=150)
plt.show()

# ── 10C: Heatmap — Val MAPE matrix ───────────────────────────────────────
numeric_matrix = val_mape_matrix.drop(
    [r for r in val_mape_matrix.index if 'naive' in r or 'persistence' in r],
    errors='ignore'
).dropna(how='all').astype(float)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    numeric_matrix,
    annot=True, fmt=".2f", cmap="RdYlGn_r",
    linewidths=0.5, ax=ax, cbar_kws={"label": "MAPE (%)"}
)
ax.set_title("Validation MAPE Heatmap — Feature Experiments × Models")
ax.set_xlabel("Model")
ax.set_ylabel("Experiment")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "val_mape_heatmap.png", dpi=150)
plt.show()

# ── 10D: Per-window MAPE over time for best config (validation) ───────────
best_pw_csv = RESULTS_DIR / f"{best_key_v[0]}_{best_key_v[1]}_per_window_val.csv"
df_best_pw  = pd.read_csv(best_pw_csv)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_best_pw["window_index"], df_best_pw["mape"],
        color="#ff7f0e", linewidth=0.8, alpha=0.85)
p90 = df_best_pw["mape"].quantile(0.90)
ax.axhline(p90, color="red", linestyle="--", linewidth=1.5,
           label=f"90th Percentile ({p90:.2f}%)")
ax.set_title(f"Per-Window MAPE over Time — Best Config ({best_key_v[0]}/{best_key_v[1]})",
             fontsize=14)
ax.set_xlabel("Window Index (Time)")
ax.set_ylabel("MAPE (%)")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "best_config_mape_over_time.png", dpi=150)
plt.show()

# ── 10E: Final test time series for best config ───────────────────────────
if "actual_te_mw" in final_result:
    n_plot = min(168, len(final_result["actual_te_mw"]))
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(final_result["actual_te_mw"][:n_plot, 0], label="True Load",
            color="black", linewidth=2)
    ax.plot(final_result["pred_te_mw"][:n_plot, 0],
            label=f"Predicted ({best_exp_name}/{best_model})",
            color="#1f77b4", linestyle="--", linewidth=1.5)
    ax.set_title(f"Test Set — Best Config Forecast (h=1, First 7 Days)", fontsize=14)
    ax.set_xlabel("Window Index")
    ax.set_ylabel("Load (MW)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "final_test_forecast.png", dpi=150)
    plt.show()

print("All plots saved to:", RESULTS_DIR)